# 03 — Detecting Intersections

Now we scale up. Instead of one polygon, we have a `FeatureCollection` of countries. Instead of asking "does this path cross this polygon?", we ask: **which features does this path cross?**

The algorithm is a double loop:

```
for each country:
    bbox check  →  skip if no overlap
    for each edge:
        segments_intersect?  →  add country to results, move on
```

The result is a list of Feature objects — the raw material for highlighting, reporting, and everything in notebook 04.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "paths.geojson") as f:
    module_paths = json.load(f)


# All intersection helpers
def orientation(p, q, r):
    return (q[0] - p[0]) * (r[1] - q[1]) - (q[1] - p[1]) * (r[0] - q[0])

def on_segment(p, q, r):
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def segments_intersect(p1, p2, p3, p4):
    d1 = orientation(p3, p4, p1)
    d2 = orientation(p3, p4, p2)
    d3 = orientation(p1, p2, p3)
    d4 = orientation(p1, p2, p4)
    if (d1 > 0 and d2 < 0 or d1 < 0 and d2 > 0) and \
       (d3 > 0 and d4 < 0 or d3 < 0 and d4 > 0):
        return True
    if d1 == 0 and on_segment(p3, p1, p4): return True
    if d2 == 0 and on_segment(p3, p2, p4): return True
    if d3 == 0 and on_segment(p1, p3, p2): return True
    if d4 == 0 and on_segment(p1, p4, p2): return True
    return False

def polygon_edges(geometry):
    if geometry["type"] == "Polygon":
        rings = geometry["coordinates"]
    elif geometry["type"] == "MultiPolygon":
        rings = [ring for poly in geometry["coordinates"] for ring in poly]
    else:
        return []
    return [(ring[i], ring[i+1]) for ring in rings for i in range(len(ring)-1)]

def segment_bbox(p1, p2):
    return (min(p1[0], p2[0]), min(p1[1], p2[1]),
            max(p1[0], p2[0]), max(p1[1], p2[1]))

def geometry_bbox(geometry):
    if geometry["type"] == "Polygon":
        rings = geometry["coordinates"]
    else:
        rings = [ring for poly in geometry["coordinates"] for ring in poly]
    coords = [pt for ring in rings for pt in ring]
    return (min(c[0] for c in coords), min(c[1] for c in coords),
            max(c[0] for c in coords), max(c[1] for c in coords))

def bboxes_overlap(b1, b2):
    return not (b1[2] < b2[0] or b2[2] < b1[0] or
                b1[3] < b2[1] or b2[3] < b1[1])

print("Helpers loaded.")

## Loading the Countries Dataset

The countries file lives in the shared `Resources/Data/` folder. It is a JSON array of Feature objects exported from a MongoDB collection — each entry has a `_id` field we strip out, and we wrap the result into a standard `FeatureCollection` saved to `data/countries.geojson` for use throughout this module.

In [ ]:
countries_path = DATA_DIR / "countries.geojson"

if not countries_path.exists():
    raw_path = Path("../../../../Resources/Data/countries_export.json")
    with open(raw_path) as f:
        raw = json.load(f)

    countries_fc = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {
                    "name": feat["properties"]["ADMIN"],
                    "iso3": feat["properties"]["ISO_A3"]
                },
                "geometry": feat["geometry"]
            }
            for feat in raw
            if feat.get("geometry") is not None
        ]
    }

    with open(countries_path, "w") as f:
        json.dump(countries_fc, f)
    print(f"Created {countries_path}")

with open(countries_path) as f:
    countries_fc = json.load(f)

print(f"{len(countries_fc['features'])} countries loaded")

## The Full Algorithm: Path Crosses Which Countries?

For a path with one or more segments, the test for a single country is:

```
bbox check fails  →  False (skip all edges)
any edge hit      →  True  (early exit — no need to check remaining edges)
all edges clear   →  False
```

Early exit matters here. Country polygons can have thousands of vertices (coastal detail, island chains). Stopping at the first intersection hit saves most of the work.

In [ ]:
def path_crosses_feature(path_feature, country_feature):
    """
    True if any segment of path_feature crosses any edge of country_feature.
    Applies bbox pre-check per path segment.
    """
    coords = path_feature["geometry"]["coordinates"]
    geom   = country_feature["geometry"]

    if geom is None:
        return False

    country_box = geometry_bbox(geom)
    edges = polygon_edges(geom)

    for i in range(len(coords) - 1):
        p1, p2 = coords[i], coords[i + 1]
        if not bboxes_overlap(segment_bbox(p1, p2), country_box):
            continue
        for e1, e2 in edges:
            if segments_intersect(p1, p2, e1, e2):
                return True

    return False


def find_crossed_countries(path_feature, countries_fc):
    """
    Return a list of country Feature objects whose borders the path crosses.
    """
    return [
        country for country in countries_fc["features"]
        if path_crosses_feature(path_feature, country)
    ]


print("Functions defined.")

## Results for All Paths

Run every path in the module dataset against every country and print the names of crossed borders.

In [ ]:
for path in module_paths["features"]:
    p = path["properties"]
    crossed = find_crossed_countries(path, countries_fc)
    names   = sorted(c["properties"]["name"] for c in crossed)
    print(f"\n{p['name']} ({p['origin']} → {p['destination']})")
    print(f"  {len(names)} countries crossed:")
    for name in names:
        print(f"    {name}")

### Why Does a Path Cross So Many Countries?

A straight `LineString` between two distant points is a **great-circle approximation on a flat projection**. The actual great-circle path curves — especially at high latitudes — and may cross different countries. For precision analysis, curved paths require more waypoints. For now, the straight-line test gives a useful first approximation of airspace impact.

## Exercise A

The **Delta** path runs from Moscow (`[37.617, 55.755]`) to Riyadh (`[46.675, 24.688]`).

1. Use `find_crossed_countries` to get the list of crossed countries.
2. Print their names in alphabetical order.
3. Print how many were skipped by the bbox pre-check vs how many required a full edge test.

For part 3, modify `path_crosses_feature` to count skips and full-tests, or add print statements temporarily.

In [ ]:
# Get the Delta path from module_paths
# 1. Find crossed countries
# 2. Print names alphabetically
# 3. Count bbox skips vs full edge tests
# Your code here

## Exercise B

Write a function `crossed_country_names(path_feature, countries_fc)` that returns a **sorted list of country name strings** (not Feature objects) for the countries a path crosses.

Then use it to answer: **which path in the dataset crosses the most countries?**

In [ ]:
def crossed_country_names(path_feature, countries_fc):
    """Return a sorted list of country name strings the path crosses."""
    # Your code here
    pass


# Use it to find which path crosses the most countries
# Your code here

---

## Check Your Understanding

**1.** Why do we stop checking edges as soon as one intersection is found, rather than continuing to find all crossing edges?

**2.** A path enters a country, travels through it, and exits the other side. How many edges does it cross at minimum, and why?

```python
# No code needed — answer in your own words
```

## Next

In [04 — Highlighting Intersected Features](./04-Highlighting_Intersected_Features.ipynb), we take the hit list and turn it into a map — styling crossed countries differently so the result is immediately visible.